# Reproducing Saez and Friedlander (2016) from CRCNS pvc-9

**Paper.** Saez I, Friedlander MJ (2016). Role of GABAA-mediated inhibition and functional assortment of
synapses onto individual layer 4 neurons in regulating plasticity expression in visual cortex. *PLoS One*
11(2): e0147642. [doi:10.1371/journal.pone.0147642](https://doi.org/10.1371/journal.pone.0147642). The
claims file (`claims/saez-2016.yaml`) was written from the version of record (PMC4739708, open access):
main text, Methods and the legends of Figures 1 to 3. The paper has no supplement and links no code.

**Data set.** CRCNS pvc-9, [doi:10.6080/K00V89R6](https://doi.org/10.6080/K00V89R6), streamed from the
public S3 bucket (`crcnsarchive/pvc-9/live/data.tar.gz`, MD5 217a64766d19e7ad87efa4e339b82295 in the data
set's `checksums.md5`). CRCNS data sets are not versioned (docs/archives.md), so the version is recorded
as `unversioned`. This is the only published copy: the bucket holds one data file, under `live/`, and no
`old/` folder; the paper cites the data set's DOI and no version. The MD5 above lets a later change be seen.

**What the data set holds.** Whole-cell recordings (Axon ABF files, 20 kHz) of layer 4 cells in guinea pig
V1 slices: extracellular stimulation at 0.2 Hz before and after a pairing protocol, in three folders,
current clamp control (`Data_ECS_IC`), voltage clamp control (`Data_ECS_VC`) and voltage clamp with
bicuculline (`Data_ECS_VCB`), one subfolder per recording. Three Excel files mark which recordings the
paper included (green) and which it excluded (red), and hold the authors' own per-sweep amplitudes and
per-cell changes. The single-cell stimulation (SCS) data of the paper's reference 17, which Figure 3 uses,
are not in it.

**Method.** The claims file fixes the per-cell measure (peak amplitude of the evoked response less a 10 ms
pre-stimulus baseline, searched from the end of the stimulus artifact to 50 ms), the pre-pairing epoch
(120 sweeps), the classification (Student's t-test p < 0.05 and a change beyond 15%), and names the open
details as readings: the post-pairing window (P) (a) all post-pairing sweeps, (b) the first 120, (c) the
last 120; and the per-cell t-test (T) (a) on sweep amplitudes, (b) on 1-min bin means. Every reading is run.
Details taken from the paper or chosen here are in `pvc9.py` and listed in the summary. No random numbers
are used.

| Claim | Tested? |
|---|---|
| dataset-size | tested |
| fig1c-control-outcomes | tested, on 20 of 28 control cells (see the data check) |
| fig1-control-no-net-plasticity | tested, on 20 of 28 |
| fig1-control-mean-change | tested, on 20 of 28 |
| fig1-vc-ic-same | tested, on 20 of 28 |
| fig1fh-control-no-dependence-on-initial | tested, on 20 of 28 |
| fig1d-example-nc | tested |
| fig2a-bicuculline-outcomes | tested |
| bicuculline-increases-plastic-fraction | tested |
| fig2-subgroup-changes | tested |
| fig2-mean-unchanged-variance-increased | tested |
| fig2cd-example-ltp-ltd | tested |
| fig2f-bicuculline-no-dependence-on-initial | tested |
| fig3a-ks-scs-vs-ecs | blocked: the SCS data of reference 17 are not public |
| fig3c-random-mixing-narrower | blocked: the SCS data of reference 17 are not public |
| fig3b-segregation-near-complete | blocked: the SCS data of reference 17 are not public |

In [ ]:
import resource
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import pvc9

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
files, extra = pvc9.stream_dataset()
sheets, members = extra["sheets"], extra["members"]
print(f"streamed {pvc9.DATA_URL} ({pvc9.DATASET}, doi:{pvc9.DATASET_DOI}) in {time.time() - started:.0f} s; "
      f"{len(members)} tar members, {len(files)} ABF files reduced, Excel files: {sorted(sheets)}")
print(f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024:.0f} MB")

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
m = members.copy()
m["ext"] = m.name.str.extract(r"\.([A-Za-z]+)$")[0].fillna("(folder)")
m["top"] = m.name.str.split("/").str[1]
print("Tar members by folder and extension:")
print(pd.crosstab(m.top.fillna(""), m.ext).to_string())
cells_dirs = m[~m.is_file & (m.name.str.count("/") == 2)].name.str.split("/").str[2]
print("\nRecording folders per group:", cells_dirs.str.extract(r"^([A-Z]+)")[0].value_counts().to_dict())
print("Descriptor: VC n=17, ICS n=21, VCB n=38.")
empty = [d for d in cells_dirs if not (files.folder == d).any()]
print("Folders without an ABF file:", empty)

The archive matches the descriptor's three folders, with 21 IC, 17 VC and 38 VCB recording folders. Besides
ABF files it holds Clampfit `.sta` files, three `.tmp` files and one `.bak` file, which the descriptor does
not mention and which are not used.

In [ ]:
f = files
print("Sampling rate (Hz):", f.rate.value_counts().to_dict())
print("Sweep length (s):", f.sweep_s.round(3).value_counts().to_dict())
print("Sweep start-to-start interval (s) by protocol:")
print(f.groupby("protocol", dropna=False).interval_s.value_counts().to_string())
print("\nUnits of channel 0 by group and protocol:")
print(pd.crosstab([f.group_dir, f.protocol.fillna("(none)")], f.units).to_string())

The descriptor gives 20 kHz and 500 ms sweeps "triggered once every 10 seconds". Most files are 20 kHz with
0.5 s sweeps, but the test sweeps are every 5 s (0.2 Hz), as the Methods state; the 10 s interval is that of
the pairing files ("Extracellular pairing", 60 trials at 0.1 Hz, Methods). The descriptor's 10 s is wrong for
the test sweeps (a finding). A few files are at 10 kHz with 1 s sweeps: they are the derived `*_PRE`/`*_POST`
files, not used. Current clamp files are in mV and voltage clamp files in pA, except the pairing files,
which are recorded in current clamp (mV), as the Methods describe.

In [ ]:
cells = pvc9.cell_table(files, sheets)
cells["n_pre_sweeps"] = [int(files.loc[r, "n_sweeps"].sum()) for r in cells.pre_rows]
cells["n_post_sweeps"] = [int(files.loc[r, "n_sweeps"].sum()) for r in cells.post_rows]
cells["files_pre"] = [" ".join(files.loc[r, "file"].str[:8]) for r in cells.pre_rows]
cells["file_pair"] = [" ".join(files.loc[r, "file"].str[:8]) for r in cells.pair_rows]
cells["files_post"] = [" ".join(files.loc[r, "file"].str[:8]) for r in cells.post_rows]
print("Recordings listed in the Summary sheets, by group and inclusion (red fill = excluded):")
print(pd.crosstab(cells.group, cells.excluded.map({False: "included", True: "excluded"})).to_string())
print("Paper: included 14 IC, 14 VC, 26 bicuculline; discarded 9 IC, 11 VC, 12 bicuculline (Methods).")
inc = cells[~cells.excluded]
print("\nIncluded recordings, with the files used for each epoch:")
print(inc[["group", "cell", "folder_exists", "source", "files_pre", "n_pre_sweeps", "file_pair", "files_post",
           "n_post_sweeps", "has_data", "rejected_files"]].to_string(index=False))
print("\nIncluded recordings with a complete baseline, pairing and post-pairing recording:")
print(inc.groupby("group").has_data.agg(["sum", "count"]).rename(columns={"sum": "complete", "count": "included"}).to_string())

In [ ]:
missing = inc[~inc.has_data]
print("Included recordings without usable post-pairing data, and the change their Summary sheet lists:")
for _, c in missing.iterrows():
    folder = files[files.folder == c.cell]
    why = ("no folder" if not c.folder_exists and not c.pre_rows else
           "no post-pairing test file" if not c.post_rows else "no pairing file")
    print(f"  {c.cell:5s} {why:28s} files in folder: {' '.join(folder.file) or '-':60s} sheet change {100 * c.sheet_change:+.1f}%")
print("\nIC sheets whose FILES block names a file whose header does not fit the epoch:")
print(inc[inc.rejected_files.str.len() > 0][["cell", "listed_files", "rejected_files"]].to_string(index=False))
for name in ["07315003.abf", "07315004.abf", "07305017.abf", "07305013.abf"]:
    r = files[files.file == name].iloc[0]
    print(f"  {r.folder}/{name}: protocol {r.protocol}, {r.n_sweeps} sweeps, interval {r.interval_s} s, units {r.units}, "
          f"start {r.start}, stimulus found at median {np.median(r.stim_ms):.1f} ms, median amplitude {np.median(r.amp50):.2f} mV")
print("\nFolder IC7 holds the files of both the IC6 and the IC7 sheets:")
print(files[files.folder == "IC7"][["folder", "file", "protocol", "n_sweeps", "start"]].to_string(index=False))

The Summary sheets mark exactly the paper's 14 IC, 14 VC and 26 bicuculline recordings as included, and
9, 11 and 12 as excluded, the Methods' discard counts. The recordings themselves are incomplete for eight
included control cells. IC22, IC23, VC21 and VC25 have no folder (the archive has folders IC1 to IC21 and
VC1 to VC17, matching the descriptor's counts of 21 and 17, which are smaller than the sheets' 23 and 25). The
folders VC10 and VC17 end with the pairing file and hold no post-pairing recording. The IC11 sheet names a
one-sweep file as the pairing and the real pairing file (60 trials) as the post-pairing recording, so its
post-pairing recording is not in the folder. The IC7 sheet names 07305017 as post-pairing; that file has no
protocol name, no recording date, a 0.5 s sweep interval and no stimulus artifact near 108 ms (its largest
step is at about 141 ms, against 107.9 ms in IC7's baseline file 07305013), so it is not a post-pairing test
recording. The folder IC6 is empty, and its files (07305007 to 07305011, as the IC6 sheet lists them) are in the
folder IC7; IC6 is analysed from them. So 10 of 14 IC, 10 of 14 VC and 26 of 26 bicuculline recordings are
complete: 20 of the 28 control cells. All of this is a finding about the data.

In [ ]:
# Files deposited more than once: the derived files (Concatenate*, *_PRE, *_POST) against the original
# recordings, by an MD5 of each sweep's samples.
orig = files[~pvc9.is_derived(files.file)]
by_md5 = {}
for i, r in orig.iterrows():
    for k, h in enumerate(r.sweep_md5):
        by_md5.setdefault(h, []).append((r.folder, r.file, k))
rows = []
for _, r in files[pvc9.is_derived(files.file)].iterrows():
    hits = [by_md5.get(h, []) for h in r.sweep_md5]
    src = sorted({f"{x[0]}/{x[1]}" for h in hits for x in h})
    rows.append(dict(folder=r.folder, file=r.file, sweeps=r.n_sweeps, units=r.units, rate=r.rate,
                     sweeps_found_in_originals=sum(bool(h) for h in hits), originals=" ".join(s.split("/")[1][:8] for s in src)))
dup = pd.DataFrame(rows)
print(dup.to_string(index=False))
same = [(k, v) for k, v in by_md5.items() if len({(x[0], x[1]) for x in v}) > 1]
print("\nSweeps shared by two original recordings:", len(same))
for h, v in same:
    print("  ", v)
dr = files[pvc9.is_derived(files.file)]
print("Derived files whose header gives 10 kHz:", dr[dr.rate == 10000].file.tolist(), "; samples per sweep:",
      sorted({int(round(x)) for x in (dr[dr.rate == 10000].sweep_s * dr[dr.rate == 10000].rate)}),
      "; originals:", sorted({int(round(x)) for x in (orig[orig.rate == 20000].sweep_s * 20000) if x == 10000}))
for h, v in same:
    for folder, fname, k in v:
        r = files[(files.folder == folder) & (files.file == fname)].iloc[0]
        print(f"  {folder}/{fname} sweep {k}: amplitude {r.amp50[k]:.2f}, baseline {r.base[k]:.2f} {r.units}")
sat = sum(int((files.loc[i, "base"] <= -1999).sum()) for c in ["pre_rows", "post_rows"]
          for rows_ in inc[inc.has_data][c] for i in rows_)
print("Sweeps pinned at -2000 pA within the analysed pre- and post-pairing epochs (all kept):", sat)

The `Concatenate*`, `*_PRE` and `*_POST` files are copies of original recordings joined end to end: their
sweeps are found in the originals one for one (all but 1 of 218 in VCB26, 80 of 215 in VCB21/Concatenate001,
an excluded cell), so each recording is counted once and only the originals are used. Four of the copies
(VCB12_POST, VCB13_PRE, VCB13_Post, VCB15_PRE) hold the same 10000 samples per sweep as the originals but their
headers give 10 kHz and 1 s sweeps, half the true rate (a finding). One sweep is identical in three
recordings of different days (VC2, VC3 and VCB19): it is pinned at -2000 pA, the amplifier's limit, with no
response, so it is not a recording deposited twice. Within the analysed epochs such sweeps are counted above
and kept, with an amplitude of 0, as the paper names no rule for dropping sweeps.

In [ ]:
used = sorted({i for col in ["pre_rows", "pair_rows", "post_rows"] for r in inc[inc.has_data][col] for i in r})
test_rows = sorted({i for col in ["pre_rows", "post_rows"] for r in inc[inc.has_data][col] for i in r})
st = np.concatenate([files.loc[i, "stim_ms"] for i in test_rows])
print(f"Stimulus time found per sweep, over {len(st)} test sweeps of the analysed cells (ms):")
print(pd.Series(np.round(st)).value_counts().sort_index().to_string())
early = [(files.loc[i, "folder"], files.loc[i, "file"], float(np.mean(files.loc[i, "stim_ms"] < 105)))
         for i in test_rows if np.mean(files.loc[i, "stim_ms"] < 105) > 0]
print("\nFiles with sweeps whose stimulus is before 105 ms (fraction of sweeps):")
for e in early:
    print(f"  {e[0]}/{e[1]}: {e[2]:.2f}")
ae = pd.Series({files.loc[i, "file"]: files.loc[i, "artifact_end_ms"] for i in test_rows})
print("\nEnd of the stimulus artifact per file (ms after the stimulus):", ae.describe().round(2).to_dict())
lat = np.concatenate([files.loc[i, "lat50"] for i in test_rows])
print(f"Peak latency (ms after stimulus): median {np.median(lat):.1f}, 5th to 95th percentile "
      f"{np.percentile(lat, 5):.1f} to {np.percentile(lat, 95):.1f}; at the window's first sample: "
      f"{np.mean(lat <= ae.min() + 0.05):.3f}")

The descriptor puts the stimulus "typically around 100 ms into the recordings". In 16709 of 17319 test sweeps
it is at 107.9 ms; in seven files 26 to 89% of the sweeps have it at 87 to 99 ms (five more files have 1 to 3%
of their sweeps so), which is why it is found per sweep; 8 sweeps give another time (50 to 78, 107, 124 or 176 ms), likely a spontaneous event larger than the
artifact. The artifact ends 0.55 to 2.6 ms after the stimulus (median 0.55), and no peak falls on the first
sample of its window.

In [ ]:
# Time-locking: the measured response after the stimulus against the same measure on a stretch of equal
# length ending 5 ms before it (where no evoked response can be), per cell, in the pre-pairing epoch.
rows = []
for _, c in inc[inc.has_data].iterrows():
    a = np.concatenate([files.loc[i, "amp50"] for i in c.pre_rows])[-pvc9.N_PRE:]
    b = np.concatenate([files.loc[i, "amp_prestim"] for i in c.pre_rows])[-pvc9.N_PRE:]
    rows.append(dict(group=c.group, cell=c.cell, after_stim=np.median(a), before_stim=np.median(b),
                     ratio=np.median(a) / np.median(b)))
tl = pd.DataFrame(rows)
print(tl.groupby("group")[["after_stim", "before_stim", "ratio"]].median().round(2).to_string())
print("Cells whose response after the stimulus is less than twice the reference:", tl[tl.ratio < 2].cell.tolist())
print("Units: IC in mV, VC and VCB in pA.")
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
t = np.arange(len(files.loc[test_rows[0], "median_trace"])) / 20.0 - 15
for ax, g in zip(axes, ["IC", "VC", "VCB"]):
    for _, c in inc[inc.has_data & (inc.group == g)].iterrows():
        tr = files.loc[c.pre_rows[-1], "median_trace"]
        tr = tr - np.median(tr[:200])
        ax.plot(t, tr, lw=0.6)
    ax.set_xlim(-15, 60)
    ax.set_title(f"{g}: median pre-pairing sweep, aligned on the stimulus")
    ax.set_xlabel("ms from stimulus")
    lims = {"IC": (-5, 15), "VC": (-700, 150), "VCB": (-800, 150)}[g]
    ax.set_ylim(*lims)
    ax.set_ylabel("mV" if g == "IC" else "pA")
plt.tight_layout()
plt.show()
print("Axes are clipped; the stimulus artifact (at 0 ms) runs beyond them in every trace.")

In every analysed cell but VCB17 the response after the stimulus is at least twice the same measure taken
before it, and the group medians are 15 to 54 times, so the evoked responses are locked to the stimulus in each
sweep (there are no trial tables; each sweep is one stimulus). VCB17 has a small pre-pairing response (about 100
pA) on a noisy holding current near -470 pA. The traces show the evoked EPSPs (IC, depolarizing) and EPSCs (VC, VCB, inward) after the
artifact.

In [ ]:
# The measure against the authors' own per-cell changes in the Summary sheets.
res = pvc9.classify_all(files, cells)
wide = res[res["T"] == "a"].pivot(index="cell", columns="P", values="change")
cmp_ = inc[inc.has_data].set_index("cell")[["group", "sheet_change", "sheet_pre"]].join(wide)
cmp_["sheet_change"] *= 100
pre_a = res[(res.P == "a") & (res["T"] == "a")].set_index("cell")
cmp_["pre_mean"] = pre_a.pre_mean
print(cmp_.round(2).to_string())
for P in "abc":
    d = (cmp_[P] - cmp_.sheet_change).abs()
    print(f"reading (P)({P}): |this change - sheet change| median {d.median():.2f} points; within 5 points: "
          f"{int((d < 5).sum())} of {len(d)}; Pearson r = {stats.pearsonr(cmp_[P], cmp_.sheet_change)[0]:.3f}")
print("\nPre-pairing amplitude, this measure against the sheet's (absolute values, VC sheets are signed):")
r_amp = stats.pearsonr(cmp_.pre_mean, cmp_.sheet_pre.abs())
print(f"Pearson r = {r_amp[0]:.3f}; median ratio {np.median(cmp_.pre_mean / cmp_.sheet_pre.abs()):.3f}")

The data set's own per-cell changes (Summary sheets, computed by the authors in Excel from Clampfit
measurements) agree with this measure for most cells (34 of 46 within 5 points under (P)(c), and the pre-pairing
amplitudes within a median 2.6%), which supports the measure and the epochs. The larger differences (IC13,
VCB11, VCB17, VCB27, and VC15 and VCB15 under (a)) may come from the sweeps the authors selected (each sheet
marks its sweeps in green) or, for VCB17, from an amplitude that includes the holding current: its sheet
gives a pre-pairing amplitude of -579 pA, close to the absolute peak on a holding current near -470 pA,
where this measure gives 98 pA. The sheet values are shown for
comparison only; every result below is computed from the recordings.

In [ ]:
print("Pre-pairing sweeps available per analysed cell (the last 120 are used; Methods, 120 stimuli):")
an = inc[inc.has_data]
print(an.groupby("group").n_pre_sweeps.describe()[["min", "50%", "max"]].to_string())
print("Cells with fewer than 120 pre-pairing sweeps:", an[an.n_pre_sweeps < 120].cell.tolist())
print("\nPost-pairing sweeps per analysed cell (Methods: at least 10 more minutes, 120 sweeps):")
print(an.groupby("group").n_post_sweeps.describe()[["min", "50%", "max"]].to_string())
print("Cells with fewer than 120 post-pairing sweeps:", an[an.n_post_sweeps < 120][["cell", "n_post_sweeps"]].values.tolist())
print("\nCells with a second pairing (only the first is used):", an[an.pair2_rows.str.len() > 0].cell.tolist())
gap = [(c.cell, (files.loc[c.post_rows[0], "start"] - files.loc[c.pair_rows[0], "start"]).total_seconds() / 60)
       for _, c in an.iterrows()]
gap = pd.DataFrame(gap, columns=["cell", "min"])
print(f"Minutes from the start of pairing to the start of the post-pairing recording: "
      f"{gap['min'].min():.1f} to {gap['min'].max():.1f} (60 trials at 0.1 Hz take 10 min)")

# Which 120 baseline sweeps: the sheets' mean pre-pairing amplitude against the first and the last 120.
rows = []
for _, c in an.iterrows():
    pre = pvc9.epoch_amplitudes(files, c.pre_rows)
    rows.append(dict(cell=c.cell, n_pre=len(pre), sheet=abs(c.sheet_pre), last120=pre[-120:].mean(), first120=pre[:120].mean()))
pw = pd.DataFrame(rows)
pw = pw[pw.n_pre > 130]
for k in ["last120", "first120"]:
    e = 100 * (pw[k] / pw.sheet - 1)
    print(f"{k}: median |difference from the sheet's pre-pairing mean| {e.abs().median():.1f}% over {len(pw)} cells with more than 130 baseline sweeps")
print("cells where the last 120 are closer:", int(((pw.last120 - pw.sheet).abs() < (pw.first120 - pw.sheet).abs()).sum()), "of", len(pw))

Every analysed cell has at least 120 pre-pairing sweeps, and most have more (up to 400). Five have fewer than
120 post-pairing sweeps (77 to 119); for them readings (P)(b) and (P)(c) use every post-pairing sweep, the
same as (a). Post-pairing recordings start 9.8 to 12.4 min after the pairing begins, as 60 trials at 0.1 Hz
require. Five IC cells had a second pairing; only the first and the recording before the second are used.

Which 120 baseline sweeps form the pre-pairing epoch is not fixed by the claims file. The Methods say pairing
began "after the pre-pairing epoch was complete (120 stimuli at 0.2 Hz)", which reads as the 120 sweeps just
before pairing, and the data set's own sheets agree: their mean pre-pairing amplitudes match the last 120
sweeps (median difference 3.8%) and not the first 120 (23.8%). The last 120 are used, as an assumption; the
first 120 are run as a variant in each classification claim.

## Claims

Every per-cell result below comes from `pvc9.classify_all`, run once for each reading of (P) and (T). The
control group is the 20 complete control recordings (10 IC, 10 VC); the bicuculline group is the 26 VCB
recordings.

In [ ]:
res = pvc9.classify_all(files, cells)
res_first = pvc9.classify_all(files, cells, pre_reading="first")      # variant: first 120 baseline sweeps
res_w30 = pvc9.classify_all(files, cells, window=30)                  # variant of (M): 30 ms window
res["control"] = res.group.isin(["IC", "VC"])
for r in (res_first, res_w30):
    r["control"] = r.group.isin(["IC", "VC"])

def counts(r, mask):
    out = r[mask].groupby(["P", "T"]).cls.value_counts().unstack(fill_value=0)
    for k in ["LTP", "LTD", "NC"]:
        if k not in out:
            out[k] = 0
    out = out[["LTP", "LTD", "NC"]]
    out["n"] = out.sum(axis=1)
    return out

print("Rows:", len(res), "=", res.cell.nunique(), "cells x 3 readings of (P) x 2 of (T)")

### dataset-size

> Plasticity was analysed in 28 control cells (14 in voltage clamp, 14 in current clamp) and 26 cells with
> 2 uM bicuculline in the bath.

Criterion: counts of the included cells, zero tolerance; the bicuculline cells are voltage clamp; 27 would be
partially reproduced; the discarded recordings (9 IC, 11 VC, 12 bicuculline) are counted where the data set
holds them.

In [ ]:
tab = pd.DataFrame({
    "marked included (Summary sheet)": cells[~cells.excluded].groupby("group").size(),
    "marked excluded": cells[cells.excluded].groupby("group").size(),
    "included, with a folder or listed files": inc[inc.folder_exists | (inc.pre_rows.str.len() > 0)].groupby("group").size(),
    "included, complete recording": inc[inc.has_data].groupby("group").size(),
})
print(tab.to_string())
print("\nPaper: 14 IC, 14 VC, 26 bicuculline included; 9 IC, 11 VC, 12 bicuculline discarded.")
print("Clamp mode of the VCB test files:", files[files.group_dir.eq("Data_ECS_VCB") &
      files.protocol.ne("Extracellular pairing")].units.value_counts().to_dict())

**Result: partially reproduced.** The Summary sheets mark exactly 14 IC, 14 VC and 26 bicuculline recordings
as included and 9, 11 and 12 as excluded, the paper's numbers, and every bicuculline test file is in voltage
clamp (pA), as the Methods and Figure 2 say and the Dataset paragraph's "I-clamp" does not. The deposit holds
complete recordings for all 26 bicuculline cells but for only 10 of the 14 IC and 10 of the 14 VC control
cells, so the control counts of what was deposited are 10 and 10, not 14 and 14.

### fig1c-control-outcomes

> With inhibition intact, few cells showed plasticity after pairing; 1 of 28 underwent LTP, 3 of 28 LTD and
> 24 of 28 no change.

Criterion: reproduced when, under one reading of (P) and (T), the counts are 1 LTP, 3 LTD, 24 NC; partially
when LTP plus LTD number 2 to 6 and NC is within 2.4 of 24. Only 20 of the 28 cells have complete recordings.
For seven of the other eight the Summary sheet lists a change below 15% in size, which makes them NC under
the paper's rule whatever the t-test gives; IC23's -24.6% leaves it LTD or NC. The counts out of 28 are
therefore given as the 20 measured cells plus 7 NC, with IC23 either way (a deviation: the classes of the
eight are bounded from the data set's own sheet values, not measured).

In [ ]:
c20 = counts(res, res.control)
print("Measured, 20 control cells:")
print(c20.to_string())
by_group = res[res.control].groupby(["P", "T", "group"]).cls.value_counts().unstack(fill_value=0)
print("\nBy clamp mode:")
print(by_group.to_string())
miss = inc[~inc.has_data & inc.group.isin(["IC", "VC"])]
forced_nc = miss[miss.sheet_change.abs() < 0.15].cell.tolist()
open_ = miss[miss.sheet_change.abs() >= 0.15][["cell", "sheet_change"]].values.tolist()
print("\nMissing control cells NC by their sheet change (|change| < 15%):", forced_nc)
print("Missing control cells that could be plastic:", open_)
rows = []
for (P, T), r in c20.iterrows():
    for ic23 in ["NC", "LTD"]:
        ltp, ltd, nc = r.LTP, r.LTD + (ic23 == "LTD"), r.NC + len(forced_nc) + (ic23 == "NC")
        exact = (ltp, ltd, nc) == (1, 3, 24)
        partial = 2 <= ltp + ltd <= 6 and abs(nc - 24) <= 2.4
        rows.append(dict(P=P, T=T, IC23=ic23, LTP=ltp, LTD=ltd, NC=nc, n=ltp + ltd + nc,
                         exact=exact, partial=partial))
b28 = pd.DataFrame(rows)
print("\nOut of 28 (20 measured + 7 NC by sheet + IC23):")
print(b28.to_string(index=False))
print("\nVariant, first 120 baseline sweeps:")
print(counts(res_first, res_first.control).to_string())
print("\nVariant, 30 ms search window:")
print(counts(res_w30, res_w30.control).to_string())
print("\nVariant, one-sided t-test (the sheets' 'p (one-tail)'):")
res_1s = pvc9.classify_all(files, cells, alternative="one-sided")
res_1s["control"] = res_1s.group.isin(["IC", "VC"])
print(counts(res_1s, res_1s.control).to_string())
plastic = res[res.control & (res.cls != "NC")]
print("\nPlastic control cells per reading:")
print(plastic.groupby(["P", "T"]).apply(lambda d: ", ".join(f"{c} {k} {x:+.1f}% p={p:.2g}" for c, k, x, p in
      zip(d.cell, d.cls, d.change, d.p))).to_string())

**Result: partially reproduced, resting on assumptions.** No reading gives 1 LTP, 3 LTD and 24 NC. On the 20
complete control cells the readings give 2 to 4 LTP and 3 to 4 LTD (6 to 8 plastic cells, 30 to 40%, against
4 of 28, 14%, in the paper). Adding the seven missing cells that the sheets put below 15% as NC, the counts out
of 28 meet the partial criterion (plastic 2 to 6, NC within 2.4 of 24) under readings (a,a), (c,a) and (c,b)
if IC23 is NC (6 plastic, 22 NC), and under none if IC23 is LTD. The outcome rests on the last 120 baseline
sweeps (with the first 120, 8 to 11 cells are plastic, mostly LTD) and on the classes of the eight missing
cells being bounded from the sheets.

### fig1-control-no-net-plasticity

> Over the 28 control cells, pairing gave no net plasticity of EPSPs or EPSCs.

Criterion: a claim of absence; a one-sample t-test of the per-cell percent changes against 0 at p < 0.05,
in each reading of (P). Reproduced when no reading reaches p < 0.05.

In [ ]:
rows = []
for P in "abc":
    x = res[res.control & (res.P == P) & (res["T"] == "a")].change.values
    t = stats.ttest_1samp(x, 0.0)
    rows.append(dict(P=P, n=len(x), mean=x.mean(), sd=x.std(ddof=1), t=t.statistic, p=t.pvalue))
    for g in ["IC", "VC"]:
        y = res[(res.group == g) & (res.P == P) & (res["T"] == "a")].change.values
        tg = stats.ttest_1samp(y, 0.0)
        rows.append(dict(P=P + " " + g, n=len(y), mean=y.mean(), sd=y.std(ddof=1), t=tg.statistic, p=tg.pvalue))
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("\nPaper: n = 28, p > 0.8 (Student's t-test).")
for P in "abc":
    x = res_first[res_first.control & (res_first.P == P) & (res_first["T"] == "a")].change.values
    print(f"Variant, first 120 baseline sweeps, ({P}): mean {x.mean():+.2f}, p = {stats.ttest_1samp(x, 0.0).pvalue:.4f}")

**Result: reproduced, resting on an assumption.** On the 20 complete control cells the mean change is -1.1% to
-1.9% and the one-sample t-test gives p = 0.54 to 0.75 in every reading of (P), so no reading finds net
plasticity (the paper's p > 0.8 is not expected to be matched; the claims file explains why). With the first
120 baseline sweeps the mean is near -11% and readings (a) and (c) reach p = 0.050 and 0.035, so the outcome
rests on the choice of the last 120. Eight of the 28 cells are missing.

### fig1-control-mean-change

> The mean change in PSR after pairing was -5.67 +/- 9.47% in voltage clamp, -1.41 +/- 10.92% in current
> clamp, and -3.53 +/- 10.26% over all 28 control cells.

Criterion: under one reading of (P), all four values within tolerance (V-clamp mean -5.67 +/- 2.53, I-clamp
mean -1.41 +/- 2.92, combined mean -3.53 +/- 1.94, combined SD 10.26 +/- 1.03); partially when the three
means are.

In [ ]:
rows = []
for P in "abc":
    r = res[res.control & (res.P == P) & (res["T"] == "a")]
    vc, ic, al = r[r.group == "VC"].change, r[r.group == "IC"].change, r.change
    vals = [vc.mean(), ic.mean(), al.mean(), al.std(ddof=1)]
    ok = [abs(vals[0] + 5.67) <= 2.53, abs(vals[1] + 1.41) <= 2.92, abs(vals[2] + 3.53) <= 1.94, abs(vals[3] - 10.26) <= 1.03]
    rows.append(dict(P=P, vc_mean=vals[0], vc_sd=vc.std(ddof=1), ic_mean=vals[1], ic_sd=ic.std(ddof=1),
                     all_mean=vals[2], all_sd=vals[3], n=len(r), within=" ".join("y" if o else "n" for o in ok)))
print(pd.DataFrame(rows).round(2).to_string(index=False))
print("\nPaper: VC -5.67 +/- 9.47, IC -1.41 +/- 10.92, all -3.53 +/- 10.26 (n = 14, 14, 28).")
sh = inc[inc.group.isin(["IC", "VC"])]
print("\nThe Summary sheets' own changes, all 28 included control cells:")
for g in ["VC", "IC"]:
    v = 100 * sh[sh.group == g].sheet_change
    print(f"  {g}: mean {v.mean():+.2f}, SD {v.std(ddof=1):.2f}, n = {v.notna().sum()}")
v = 100 * sh.sheet_change
print(f"  all: mean {v.mean():+.2f}, SD {v.std(ddof=1):.2f}, n = {v.notna().sum()}")
for P in "abc":
    r = res_first[res_first.control & (res_first.P == P) & (res_first["T"] == "a")]
    print(f"Variant, first 120 baseline sweeps, ({P}): VC {r[r.group == 'VC'].change.mean():+.2f}, "
          f"IC {r[r.group == 'IC'].change.mean():+.2f}, all {r.change.mean():+.2f} +/- {r.change.std(ddof=1):.2f}")
v20 = 100 * sh[sh.has_data].sheet_change
print(f"  the 20 complete cells, sheet values: mean {v20.mean():+.2f}, SD {v20.std(ddof=1):.2f}")

**Result: partially reproduced, resting on an assumption.** Under reading (P)(c) the three means are within
tolerance (VC -4.36%, IC +0.46%, all -1.95%, against -5.67 +/- 2.53, -1.41 +/- 2.92 and -3.53 +/- 1.94) but
the SD over cells is 14.13%, outside 10.26 +/- 1.03; readings (a) and (b) miss the combined mean as well. These
are means over 10 VC and 10 IC cells, not 14 and 14. With the first 120 baseline sweeps every mean is near -9
to -14% and none is within tolerance. The sheets' own changes for all 28 cells give VC -3.14 +/- 13.40% and IC
-5.79 +/- 9.51%: the sheets' IC values are close to the paper's VC values and the reverse is not so close.

### fig1-vc-ic-same

> Voltage-clamp and current-clamp control outcomes did not differ; both were Gaussian, and their means and
> variances did not differ, so the two were combined.

Criterion: claims of absence; Shapiro-Wilk on each group, a two-sample Student's t-test and a two-sided
F-test of the variances, at p < 0.05, in each reading of (P). Reproduced when no reading has any test reach
p < 0.05; not reproduced when every reading has one that does.

In [ ]:
def ftest(x, y):
    """Two-sided F-test of the ratio of variances, var(x) / var(y)."""
    F = np.var(x, ddof=1) / np.var(y, ddof=1)
    p = 2 * min(stats.f.cdf(F, len(x) - 1, len(y) - 1), stats.f.sf(F, len(x) - 1, len(y) - 1))
    return F, min(p, 1.0)

rows = []
for P in "abc":
    r = res[res.control & (res.P == P) & (res["T"] == "a")]
    vc, ic = r[r.group == "VC"].change.values, r[r.group == "IC"].change.values
    F, pF = ftest(vc, ic)
    rows.append(dict(P=P, shapiro_vc=stats.shapiro(vc).pvalue, shapiro_ic=stats.shapiro(ic).pvalue,
                     t_p=stats.ttest_ind(vc, ic).pvalue, mean_diff=vc.mean() - ic.mean(), F=F, F_p=pF))
out = pd.DataFrame(rows)
out["any_p_below_0.05"] = out[["shapiro_vc", "shapiro_ic", "t_p", "F_p"]].min(axis=1) < 0.05
print(out.round(4).to_string(index=False))
print("\nPaper: Shapiro-Wilk p = 0.79 (VC), 0.85 (IC); t-test p = 0.28; F-test p = 0.61.")
for P in "abc":
    r = res_first[res_first.control & (res_first.P == P) & (res_first["T"] == "a")]
    vc, ic = r[r.group == "VC"].change.values, r[r.group == "IC"].change.values
    print(f"Variant, first 120 baseline sweeps, ({P}): Shapiro VC p = {stats.shapiro(vc).pvalue:.3f}, IC p = {stats.shapiro(ic).pvalue:.3f}, "
          f"t p = {stats.ttest_ind(vc, ic).pvalue:.3f}, F p = {ftest(vc, ic)[1]:.3f}")

**Result: reproduced.** In every reading of (P) none of the four tests reaches p < 0.05: Shapiro-Wilk p =
0.20 to 0.85 (VC) and 0.15 to 0.49 (IC), t-test p = 0.46 to 0.63, F-test p = 0.35 to 0.83, on 10 cells per
mode; the first-120 variant gives the same (all p above 0.2).

### fig1fh-control-no-dependence-on-initial

> In control cells the plasticity outcome did not depend on the initial synaptic strength, in current clamp
> or in voltage clamp.

Criterion: Pearson correlation of the pre-pairing mean amplitude against the percent change, per clamp mode,
at p < 0.05 (Spearman as a variant). Reproduced when neither mode reaches p < 0.05. Judged in each reading
of (P), as the claims file's rule for absence says.

In [ ]:
rows = []
for P in "abc":
    for g in ["IC", "VC"]:
        r = res[(res.group == g) & (res.P == P) & (res["T"] == "a")]
        pr, sr = stats.pearsonr(r.pre_mean, r.change), stats.spearmanr(r.pre_mean, r.change)
        rows.append(dict(P=P, group=g, n=len(r), pearson_r=pr[0], pearson_p=pr[1], spearman_rho=sr[0], spearman_p=sr[1],
                         initial_range=f"{r.pre_mean.min():.1f} to {r.pre_mean.max():.1f}"))
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("Initial amplitude in mV (IC) and pA (VC). Paper: IC p > 0.9, VC p > 0.1.")
for P in "abc":
    for g in ["IC", "VC"]:
        r = res_first[(res_first.group == g) & (res_first.P == P) & (res_first["T"] == "a")]
        print(f"Variant, first 120 baseline sweeps, ({P}) {g}: Pearson p = {stats.pearsonr(r.pre_mean, r.change)[1]:.3f}")
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
for ax, g in zip(axes, ["IC", "VC"]):
    r = res[(res.group == g) & (res.P == "a") & (res["T"] == "a")]
    ax.scatter(r.pre_mean, r.change, s=14, c=r.cls.map({"LTP": "tab:red", "LTD": "tab:blue", "NC": "grey"}))
    ax.axhline(15, ls="--", c="grey", lw=0.6); ax.axhline(-15, ls="--", c="grey", lw=0.6)
    ax.set_xlabel("pre-pairing amplitude (" + ("mV" if g == "IC" else "pA") + ")"); ax.set_ylabel("change (%)")
    ax.set_title(f"{g} control, reading (P)(a)")
plt.tight_layout(); plt.show()

**Result: reproduced.** The Pearson correlation of initial amplitude and change does not reach p < 0.05 in
either mode in any reading (IC p = 0.21 to 0.29, r = -0.38 to -0.43; VC p = 0.10 to 0.25, r = 0.40 to 0.55),
nor with Spearman's correlation or the first 120 baseline sweeps, on 10 cells per mode. The paper's IC p > 0.9
is not matched; with 10 cells the IC correlation is moderate and negative.

### fig1d-example-nc

> A representative current-clamp control cell showed no plasticity (EPSP change -1 +/- 0.1%, p = 0.48).

Criterion: reproduced when, under one reading of (P) and (T), at least one included I-clamp control cell has
a change within -1 +/- 0.1% and a t-test p of 0.05 or more; the closest cell is reported either way.

In [ ]:
ic = res[res.group == "IC"].copy()
ic["dist"] = (ic.change + 1.0).abs()
best = ic.sort_values("dist").groupby(["P", "T"]).head(1)
print(best[["P", "T", "cell", "change", "p", "cls"]].round(4).to_string(index=False))
hit = ic[(ic.dist <= 0.1) & (ic.p >= 0.05)]
icf = res_first[res_first.group == "IC"]
print("Variant, first 120 baseline sweeps, IC6:", icf[icf.cell == "IC6"][["P", "T", "change", "p"]].round(3).values.tolist())
print("\nIC cells within -1 +/- 0.1% with p >= 0.05:")
print(hit[["P", "T", "cell", "change", "p"]].round(4).to_string(index=False) if len(hit) else "none")

**Result: reproduced, resting on an assumption.** IC6 changes by -0.94% with t-test p = 0.47 under (P)(a) and
(T)(a), and by -1.07% with p = 0.41 under (P)(c): within -1 +/- 0.1% with p of 0.05 or more, close to the
paper's p = 0.48. With the first 120 baseline sweeps IC6 changes by -22% and no IC cell matches.

### fig2a-bicuculline-outcomes

> With GABAA receptors blocked by 2 uM bicuculline, more cells changed after pairing (16 of 26), with
> outcomes of both signs; 7 of 26 underwent LTP, 9 of 26 LTD and 10 of 26 no change.

Criterion: reproduced when, under one reading of (P) and (T), the counts are 7, 9 and 10; partially when LTP
and LTD are each within 2 of the paper and both are present.

In [ ]:
cb = counts(res, res.group == "VCB")
cb["exact"] = (cb.LTP == 7) & (cb.LTD == 9) & (cb.NC == 10)
cb["partial"] = ((cb.LTP - 7).abs() <= 2) & ((cb.LTD - 9).abs() <= 2) & (cb.LTP > 0) & (cb.LTD > 0)
print(cb.to_string())
print("\nVariant, first 120 baseline sweeps:")
print(counts(res_first, res_first.group == "VCB").to_string())
print("\nVariant, 30 ms search window:")
print(counts(res_w30, res_w30.group == "VCB").to_string())
print("\nVariant, one-sided t-test:")
print(counts(res_1s, res_1s.group == "VCB").to_string())
cls = res[res.group == "VCB"].pivot_table(index="cell", columns=["P", "T"], values="cls", aggfunc="first")
cls.columns = [f"class_{p}{t}" for p, t in cls.columns]
chg = res[(res.group == "VCB") & (res["T"] == "a")].pivot(index="cell", columns="P", values="change").round(1)
chg.columns = [f"change_{c}" for c in chg.columns]
print("\nClass of each bicuculline cell per reading, with its change under each (P):")
print(cls.join(chg).to_string())

**Result: partially reproduced, resting on an assumption.** No reading gives exactly 7 LTP, 9 LTD and 10 NC.
Readings (a,a) (8, 8, 10), (a,b) (7, 7, 12), (c,a) and (c,b) (7, 8, 11) have LTP and LTD each within 2 of the
paper and both present; reading (b) has too few LTD (5 or 6). Every reading has 13 to 16 plastic cells of 26,
of both signs. The 30 ms window and the one-sided test give the same counts; the first 120 baseline sweeps give
6 LTP and 11 to 14 LTD, outside the partial criterion.

### bicuculline-increases-plastic-fraction

> Blocking GABAA inhibition raised the fraction of cells with plasticity from 4 of 28 to 16 of 26, showing
> that inhibition limits plasticity induction onto L4 cells.

Criterion: reproduced when, under the same reading for both groups, the bicuculline fraction of plastic
cells is larger than the control fraction and both counts are within tolerance (4 +/- 0.4, 16 +/- 1.6);
partially when the fraction is larger but a count is outside tolerance. Fisher's exact test as a variant.
The control count is out of the 20 measured cells and, as in fig1c-control-outcomes, out of 28 with the
eight missing cells bounded from their sheet values.

In [ ]:
rows = []
for (P, T), r in c20.iterrows():
    pc = r.LTP + r.LTD
    rb = cb.loc[(P, T)]
    pb = rb.LTP + rb.LTD
    for ic23 in ["NC", "LTD"]:
        pc28 = pc + (ic23 == "LTD")
        odds, pf = stats.fisher_exact([[pb, 26 - pb], [pc28, 28 - pc28]])
        rows.append(dict(P=P, T=T, IC23=ic23, control_plastic=f"{pc}/20, {pc28}/28", control_frac=pc28 / 28,
                         bic_plastic=f"{pb}/26", bic_frac=pb / 26, larger=pb / 26 > pc28 / 28,
                         control_in_tol=abs(pc28 - 4) <= 0.4, bic_in_tol=abs(pb - 16) <= 1.6, fisher_p=pf))
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("\nOn the 20 measured control cells alone:")
for (P, T), r in c20.iterrows():
    pc, pb = r.LTP + r.LTD, cb.loc[(P, T)].LTP + cb.loc[(P, T)].LTD
    print(f"  ({P},{T}) control {pc}/20 = {pc / 20:.2f}, bicuculline {pb}/26 = {pb / 26:.2f}, "
          f"Fisher p = {stats.fisher_exact([[pb, 26 - pb], [pc, 20 - pc]])[1]:.4f}")

**Result: partially reproduced, resting on assumptions.** In every reading the plastic fraction is larger in
bicuculline (13 to 16 of 26, 50 to 62%) than in control (6 to 8 of the 20 measured, 6 to 9 of 28 with the
missing cells bounded). The bicuculline count is within 16 +/- 1.6 under (a,a), (c,a) and (c,b), but the
control count is never within 4 +/- 0.4. Fisher's exact test gives p = 0.005 to 0.27 against the bounded 28
(p = 0.042 to 0.56 against the 20 measured), significant at 0.05 under (a,a), (c,a), (c,b) and, with IC23 NC,
(a,b) and (b,a).

### fig2-subgroup-changes

> In bicuculline, LTP cells changed by +39.9 +/- 14.7%, LTD cells by -38.5 +/- 20.4%, and NC cells by
> -3.0 +/- 7.6% on average.

Criterion: means over the cells of each class, under the same reading as fig2a-bicuculline-outcomes;
tolerances 5.56, 6.8 and 2.4. Reproduced when all three are within tolerance under one reading; partially
when two are.

In [ ]:
rows = []
exp = {"LTP": (39.9, 5.56), "LTD": (-38.5, 6.8), "NC": (-3.0, 2.4)}
for (P, T), r in res[res.group == "VCB"].groupby(["P", "T"]):
    row = dict(P=P, T=T)
    k = 0
    for c, (v, tol) in exp.items():
        x = r[r.cls == c].change
        row[f"{c}_n"] = len(x)
        row[f"{c}_mean"] = x.mean()
        row[f"{c}_sd"] = x.std(ddof=1)
        ok = abs(x.mean() - v) <= tol if len(x) else False
        row[f"{c}_ok"] = ok
        k += ok
    row["within"] = k
    rows.append(row)
print(pd.DataFrame(rows).round(2).to_string(index=False))
print("\nPaper: LTP +39.9 +/- 14.7 (n = 7), LTD -38.5 +/- 20.4 (n = 9), NC -3.0 +/- 7.6 (n = 10).")
print("\nVariant, without VCB17 (change near +400%, see the data check):")
for (P, T), r in res[(res.group == "VCB") & (res.cell != "VCB17")].groupby(["P", "T"]):
    x = r[r.cls == "LTP"].change
    print(f"  ({P},{T}) LTP n = {len(x)}, mean {x.mean():+.2f}, SD {x.std(ddof=1):.2f}, within 5.56 of 39.9: {abs(x.mean() - 39.9) <= 5.56}")
print("\nVariant, first 120 baseline sweeps:")
for (P, T), r in res_first[res_first.group == "VCB"].groupby(["P", "T"]):
    print(f"  ({P},{T}) " + ", ".join(f"{c} n = {(r.cls == c).sum()} mean {r[r.cls == c].change.mean():+.1f}" for c in exp))

**Result: partially reproduced, resting on an assumption.** Under (a,a), (a,b), (c,a) and (c,b) the LTD mean
(-35.2 to -36.9%) and the NC mean (-4.7 to -5.0%) are within tolerance and the LTP mean (+80 to +89%) is not.
The LTP mean is raised by VCB17, whose change is near +400% with this measure (+63% in its sheet); without it
the LTP mean is +34.4 to +38.2%, within tolerance in every reading shown.

### fig2-mean-unchanged-variance-increased

> The mean change over all cells did not differ between bicuculline (-3.8 +/- 34.3%) and control (-3.5 +/-
> 10.3%), but the variance of plasticity outcomes was significantly larger in bicuculline.

Criterion: two-sample Student's t-test of the means, judged as absence in each reading of (P); a two-sided
F-test of the variances with p < 0.01 and the bicuculline variance larger, in the same reading; the
bicuculline mean (-3.8 +/- 6.73) and SD (34.3 +/- 3.43) within tolerance. Reproduced when all hold; partially
when the two tests hold but a value is outside tolerance.

In [ ]:
rows = []
for P in "abc":
    b = res[(res.group == "VCB") & (res.P == P) & (res["T"] == "a")].change.values
    c = res[res.control & (res.P == P) & (res["T"] == "a")].change.values
    F, pF = ftest(b, c)
    rows.append(dict(P=P, n_bic=len(b), bic_mean=b.mean(), bic_sd=b.std(ddof=1), n_ctl=len(c), ctl_mean=c.mean(),
                     ctl_sd=c.std(ddof=1), t_p=stats.ttest_ind(b, c).pvalue, F=F, F_p=pF,
                     mean_in_tol=abs(b.mean() + 3.8) <= 6.73, sd_in_tol=abs(b.std(ddof=1) - 34.3) <= 3.43))
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("\nPaper: bicuculline -3.8 +/- 34.3 (n = 26), control -3.5 +/- 10.3 (n = 28); t-test p = 0.97; Fisher's test p < 0.01.")
sb = 100 * inc[inc.group == "VCB"].sheet_change
print(f"The VCB Summary sheet's own changes: mean {sb.mean():+.2f}, SD {sb.std(ddof=1):.2f}, n = {sb.notna().sum()}")
print("\nVariant, without VCB17:")
for P in "abc":
    b = res[(res.group == "VCB") & (res.cell != "VCB17") & (res.P == P) & (res["T"] == "a")].change.values
    c = res[res.control & (res.P == P) & (res["T"] == "a")].change.values
    F, pF = ftest(b, c)
    print(f"  ({P}) n = {len(b)}, mean {b.mean():+.2f}, SD {b.std(ddof=1):.2f}, t-test p = {stats.ttest_ind(b, c).pvalue:.3f}, "
          f"F = {F:.2f}, F-test p = {pF:.2g}")
print("Variant, first 120 baseline sweeps:")
for P in "abc":
    b = res_first[(res_first.group == "VCB") & (res_first.P == P) & (res_first["T"] == "a")].change.values
    c = res_first[res_first.control & (res_first.P == P) & (res_first["T"] == "a")].change.values
    F, pF = ftest(b, c)
    print(f"  ({P}) mean {b.mean():+.2f}, SD {b.std(ddof=1):.2f}, t-test p = {stats.ttest_ind(b, c).pvalue:.3f}, F-test p = {pF:.2g}")
fig, ax = plt.subplots(figsize=(6, 3.2))
bins = np.arange(-100, 105, 10)
for g, col in [("control", "grey"), ("VCB", "tab:green")]:
    x = res[(res.control if g == "control" else res.group == "VCB") & (res.P == "a") & (res["T"] == "a")].change
    ax.hist(np.clip(x, -100, 100), bins=bins, alpha=0.6, color=col, label=f"{g} (n = {len(x)})")
ax.set_xlabel("change after pairing (%), reading (P)(a)"); ax.legend()
plt.tight_layout(); plt.show()
x = res[(res.P == "a") & (res["T"] == "a")].change
print(f"Changes clipped to +/-100% in the plot: {int((x.abs() > 100).sum())} cell(s), largest {x.abs().max():.1f}%")

**Result: partially reproduced.** In every reading of (P) the means do not differ (t-test p = 0.45 to 0.52) and
the bicuculline variance is larger (F = 29 to 36, p < 0.01), so both tests hold; but the bicuculline mean
(+10.4 to +13.5%) and SD (84 to 85%) are outside -3.8 +/- 6.73 and 34.3 +/- 3.43, because of VCB17 (+400%).
Without VCB17 the mean is -1.8 to -5.1% and the SD 29.1 to 31.6%, close to the paper; the tests still hold
(t-test p = 0.66 to 0.93, F-test p = 0.001 to 0.003). The control group is 20 of 28 cells. The first 120
baseline sweeps give the same tests (t-test p of 0.69 to 0.79, F-test p of 0.0015 to 0.0048).

### fig2cd-example-ltp-ltd

> Representative bicuculline cells underwent LTP (+59.4 +/- 26%, p < 0.001) and LTD (-37.6 +/- 19.9%,
> p < 0.001).

Criterion: reproduced when, under one reading of (P) and (T), one included bicuculline cell has a change
within 59.4 +/- 5.94% and another within -37.6 +/- 3.76%, each with p < 0.001; partially when one is found.

In [ ]:
vb = res[res.group == "VCB"].copy()
for name, (v, tol) in {"LTP": (59.4, 5.94), "LTD": (-37.6, 3.76)}.items():
    vb["dist"] = (vb.change - v).abs()
    print(f"{name} example ({v} +/- {tol}%), closest cell per reading:")
    print(vb.sort_values("dist").groupby(["P", "T"]).head(1)[["P", "T", "cell", "change", "p"]].to_string(index=False))
    h = vb[(vb.dist <= tol) & (vb.p < 0.001)]
    print("  matches with p < 0.001:", sorted(set(zip(h.cell, h.P, h["T"]))))
    rf = res_first[res_first.group == "VCB"]
    hf = rf[((rf.change - v).abs() <= tol) & (rf.p < 0.001)]
    print("  variant, first 120 baseline sweeps, matches:", sorted(set(zip(hf.cell, hf.P, hf["T"]))))
ex = []
for cell in ["VCB29", "VCB7", "VCB10"]:
    c = inc[inc.cell == cell].iloc[0]
    ep = pvc9.cell_epochs(files, c)
    ex.append(dict(cell=cell, pre_mean=ep["pre"].mean(), pre_sd=ep["pre"].std(ddof=1),
                   post_mean=ep["post"]["a"].mean(), post_sd=ep["post"]["a"].std(ddof=1),
                   sweep_sd_pct_of_pre=100 * ep["post"]["a"].std(ddof=1) / ep["pre"].mean()))
print("\nSpread over sweeps of the closest cells (pA), reading (P)(a):")
print(pd.DataFrame(ex).round(1).to_string(index=False))

**Result: reproduced.** VCB29 changes by +60.1% (p = 2.6e-65 under (a,a)), within 59.4 +/- 5.94%, and VCB10
by -36.9% (p = 1.4e-34) and VCB7 by -36.4% under (P)(a), within -37.6 +/- 3.76%, all with p < 0.001. The
sweep-to-sweep SDs of these cells (19 to 26% of the pre-pairing mean) are of the order of the paper's +/- 26
and 19.9%. The match holds with the first 120 baseline sweeps as well.

### fig2f-bicuculline-no-dependence-on-initial

> In bicuculline the plasticity outcome did not depend on the initial synaptic strength.

Criterion: Pearson correlation of the pre-pairing mean EPSC against the percent change, p < 0.05 (Spearman
as a variant). Reproduced when p is 0.05 or more, judged in each reading of (P).

In [ ]:
rows = []
for P in "abc":
    r = res[(res.group == "VCB") & (res.P == P) & (res["T"] == "a")]
    pr, sr = stats.pearsonr(r.pre_mean, r.change), stats.spearmanr(r.pre_mean, r.change)
    rows.append(dict(P=P, n=len(r), pearson_r=pr[0], pearson_p=pr[1], spearman_rho=sr[0], spearman_p=sr[1]))
print(pd.DataFrame(rows).round(4).to_string(index=False))
r = res[(res.group == "VCB") & (res.P == "a") & (res["T"] == "a")]
print(f"Initial EPSC {r.pre_mean.min():.1f} to {r.pre_mean.max():.1f} pA. Paper: p > 0.5 (Results), p > 0.3 (legend).")
for P in "abc":
    r = res_first[(res_first.group == "VCB") & (res_first.P == P) & (res_first["T"] == "a")]
    print(f"Variant, first 120 baseline sweeps, ({P}): Pearson p = {stats.pearsonr(r.pre_mean, r.change)[1]:.3f}")

**Result: reproduced.** The Pearson correlation of initial EPSC and change over the 26 bicuculline cells is
r = -0.15 to -0.16, p = 0.44 to 0.48 in every reading (Spearman p = 0.79 to 0.95); p is above 0.3, the legend's
bound, and just below the Results' 0.5.

### fig3a-ks-scs-vs-ecs, fig3c-random-mixing-narrower, fig3b-segregation-near-complete

These three claims need the single-cell stimulation (SCS) data of the paper's reference 17 (per-connection
strengths before and after pairing, n = 42 or 43). The data set does not hold them (the archive's listing
above has only the three ECS folders and their sheets), the paper gives no location, and no CRCNS data set
of these authors holds them (the bucket's `pvc-1` to `pvc-13` were listed while preparing this entry; only
`pvc-9` is theirs). They are blocked as `data_missing`.

In [ ]:
print("Archive folders:", sorted(members.name.str.split("/").str[1].dropna().unique()))
print("Excel sheets:", {k: len(v) for k, v in sheets.items()}, "bytes")

**Result: blocked (data missing).** The SCS data of reference 17, which all three Figure 3 claims need, are not
in the data set and not public anywhere found.

## Summary

| Claim | Outcome |
|---|---|
| dataset-size | partially reproduced |
| fig1c-control-outcomes | partially reproduced (assumed) |
| fig1-control-no-net-plasticity | reproduced (assumed) |
| fig1-control-mean-change | partially reproduced (assumed) |
| fig1-vc-ic-same | reproduced |
| fig1fh-control-no-dependence-on-initial | reproduced |
| fig1d-example-nc | reproduced (assumed) |
| fig2a-bicuculline-outcomes | partially reproduced (assumed) |
| bicuculline-increases-plastic-fraction | partially reproduced (assumed) |
| fig2-subgroup-changes | partially reproduced (assumed) |
| fig2-mean-unchanged-variance-increased | partially reproduced |
| fig2cd-example-ltp-ltd | reproduced |
| fig2f-bicuculline-no-dependence-on-initial | reproduced |
| fig3a-ks-scs-vs-ecs | blocked: data missing |
| fig3c-random-mixing-narrower | blocked: data missing |
| fig3b-segregation-near-complete | blocked: data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

**How the analysis differs from the paper's.** The paper measured amplitudes in Clampfit, on sweeps it selected,
and gives no peak window, baseline, post-pairing window or form of t-test. Here the claims file's measure is
used on every sweep, with the stimulus found per sweep and the end of the artifact found per file; the
post-pairing window and the t-test are run under every reading. The pre-pairing epoch is the last 120 baseline
sweeps, an assumption supported by the Methods' wording and by the data set's own sheets; the first 120 change
several outcomes and are shown as a variant. Cells with two pairings use the first. The control group is 20
of the paper's 28 cells, as 8 have no complete recording.

**What was found about the data set.**
- Eight of the 28 included control recordings are missing or incomplete: IC22, IC23, VC21 and VC25 have no
  folder; VC10 and VC17 have no post-pairing recording; IC11's post-pairing recording is not in its folder
  (its sheet names the pairing file as the post-pairing one); IC7's listed post-pairing file (07305017) has
  no protocol, no date and no evoked response at the usual time.
- Folder IC6 is empty and the IC6 recording is in folder IC7, with IC7's own.
- The descriptor says sweeps are "triggered once every 10 seconds"; test sweeps are every 5 s (0.2 Hz), as the
  Methods say; only pairing trials are every 10 s.
- The derived files (`Concatenate*`, `*_PRE`, `*_POST`) duplicate the original recordings; four give half the
  true sampling rate in their headers. The archive also holds `.sta`, `.tmp` and `.bak` files that the
  descriptor does not mention.
- The stimulus falls 9 to 20 ms earlier in some sweeps of 12 files.
- The Summary sheets' per-cell changes reproduce the paper's bicuculline mean and SD (-3.76 +/- 34.25%), but
  their IC and VC values (-5.79 +/- 9.51%, -3.14 +/- 13.40%) do not match the paper's I-clamp and V-clamp values
  (-1.41 +/- 10.92%, -5.67 +/- 9.47%); the IC values are close to the paper's V-clamp ones.
- For VCB17 the sheet's amplitude appears to include the holding current (about -579 pA against a response near
  100 pA), which turns a +400% change into +63%.